In [1]:
!pip install -q playwright
!python -m playwright install chromium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 MB 38.1 MB/s eta 0:00:00
(node:57) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
164.7 MiB [                    ] 0% 339.3s164.7 MiB [                    ] 0% 24.6s164.7 MiB [                    ] 0% 11.3s164.7 MiB [                    ] 0% 8.3s164.7 MiB [                    ] 1% 4.5s164.7 MiB [                    ] 2% 3.5s164.7 MiB [=                   ] 3% 2.9s164.7 MiB [=                   ] 4% 2.8s164.7 MiB [=                   ] 4% 2.7s164.7 MiB [=                   ] 5% 2.4s164.7 MiB [=                   ] 6% 2.2s164.7 MiB [==                  ] 7% 2.1s164.7 MiB [==                  ] 9% 2.0s164.7 MiB [==                  ] 10% 1.9s164.7 MiB [==                  ] 11% 1.8s164.7 MiB [===   

In [2]:
from playwright.async_api import async_playwright
import time
import requests
from bs4 import BeautifulSoup
import re
from datetime import date, timedelta
from urllib.parse import urlencode
import asyncio, aiohttp

import warnings
warnings.filterwarnings("ignore")


# input

In [3]:
from tqdm.asyncio import tqdm_asyncio

async def fetch(session, path):
    try:
        async with session.get(path) as r:
            html = await r.text()
        soup = BeautifulSoup(html, "lxml")
        return [
            [td.get_text("\n", strip=True) for td in tr.select("td") if td.text.strip()]
            for tr in soup.select("tr")
        ]
    except Exception as e:
        return e

async def main(paths, limit=30):
    async with aiohttp.ClientSession(
        base_url="https://www.boatrace.jp",
        connector=aiohttp.TCPConnector(limit=limit),
        headers={"User-Agent": "Mozilla/5.0"}
    ) as session:
        results = await tqdm_asyncio.gather(
            *(fetch(session, p) for p in paths)
        )
    return results


In [4]:
import asyncio, aiohttp
from bs4 import BeautifulSoup

BASE = "https://example.com/"

async def fetch(session, path):
    async with session.get(BASE + path) as r:
        html = await r.text()
    soup = BeautifulSoup(html, "lxml")
    return [
        [td.get_text("\n", strip=True) for td in tr.select("td") if td.text.strip()]
        for tr in soup.select("tr")
    ]

async def main(paths, limit=30):
    connector = aiohttp.TCPConnector(limit=limit)
    async with aiohttp.ClientSession(
        connector=connector,
        headers={"User-Agent": "Mozilla/5.0"}
    ) as session:
        return await asyncio.gather(*(fetch(session, p) for p in paths))


# get_racelist

In [5]:
def get_racelist(texts):
    texts = [text[1:7] for text in texts if len(text)>=8]
    res = {}
    rank = {'A1': 4, 'A2': 3, 'B1': 2, 'B2': 1}
    
    for i, row in enumerate(texts):
        s = row[0]
        s = list(map(str, s.split('\n')))
        player_id = s[0]
        grade = rank[s[2]]
        age = int(s[5].split('/')[0][:-1])
        weight = float(s[5].split('/')[1][:-2])
        
        f_l = row[1]
        f = int(re.search(r"F(\d+)", f_l).group(1))
        l = int(re.search(r"L(\d+)", f_l).group(1))
        st = float(re.search(r"(\d+\.\d+)", f_l).group(1))
        avg1, rate1, rate2 = map(float, row[2].split('\n'))
        avg2, rate3, rate4 = map(float, row[3].split('\n'))
        _, mot1, mot2 = map(float, row[4].split('\n'))
        _, bot1, bot2 = map(float, row[5].split('\n'))

    # return [grade,age,weight,f,l,st,avg1,rate1,
    #         rate2,avg2,rate3,rate4,mot1,mot2,bot1,bot2]
    
        res[i+1] = {
            "player_id": player_id,
            "grade": grade,
            "age": age,
            "weight": weight,
            "f": f,
            "l": l,
            "st": st,
            "avg1": avg1,
            "rate1": rate1,
            "rate2": rate2,
            "avg2": avg2,
            "rate3": rate3,
            "rate4": rate4,
            "mot1": mot1,
            "mot2": mot2,
            "bot1": bot1,
            "bot2": bot2,
        }
    return res

# odds_total

In [6]:
def odds3t(texts):
    ODDS = {}
    texts = [t for text in texts for t in text]
    texts = map(str, texts)
    
    for _ in range(5):
        A = []
        for i in range(1, 7):
            a, b = int(next(texts)), int(next(texts))
            c = float(next(texts))
            assert (i, a, b) not in ODDS
            ODDS[1, i, a, b] = c
            A.append(a)
        for _ in range(3):
            for i in range(1, 7):
                b, c = int(next(texts)), float(next(texts))
                ODDS[1, i, A[i-1], b] = c
    return ODDS

def odds3f(texts):
    ODDS = {}
    # print(texts)
    for text in texts:
        if len(text)==3 or (len(text)>3 and text[0]==text[3]):
            a = int(text[0])
            text = [text[i] for i in range(len(text)) if i%3!=0]
        for b in range(1, len(text)//2+1):
            c = int(text[2*(b-1)])
            ODDS[2, b, a, c] = text[2*(b-1)+1]

    return ODDS

def odds2tf(texts):
    ODDS = {}
    for text in texts[:5]:
        for a in range(6):
            b, c = text[2*a:2*a+2]
            ODDS[(3, a+1, int(b))] = float(c)
    for text in texts[6:]:
        for a in range(len(text)//2):
            b, c = text[2*a:2*a+2]
            ODDS[(4, a+1, int(b))] = float(c)
    return ODDS

def oddsk(texts):
    ODDS = {}
    for text in texts[:5]:
        for a in range(len(text)//2):
            b, c = text[2*a:2*a+2]
            c, d = map(float, c.split('-'))
            ODDS[5, a+1, int(b)] = (c+d)/2
    return ODDS

def oddstf(texts):
    ODDS = {}
    for text in texts[:6]:
        a, b = text[0], text[2]
        if b=='欠場': return False
        ODDS[6, int(a)] = float(b)
    for text in texts[7:]:
        a, b = text[0], text[2]
        c, d = map(float, b.split('-'))
        ODDS[7, int(a)] = float(c)
    return ODDS

def odds_total(TEXTS):
    ODDS = oddstf(TEXTS[0][3:])
    if ODDS==False: return False
    for texts, f in zip(TEXTS[1:], [odds3t, odds3f, odds2tf, oddsk]):
        ODDS |= f(texts[3:])

    return ODDS

# get_result

In [7]:
def get_result(rows):
    res = []
    tof = True
    for row in rows:
        if not row:
            continue
        if row[0]=="3連単":
            return tuple(map(int, re.findall(r"\d+", row[1])))

In [8]:
def make_date(days=5):
    end = date(2026, 1, 3)
    start = end - timedelta(days=days)
    
    dates = []
    
    d = start
    while d < end:
        for jcd in range(1, 13):
            for rno in range(1, 25):
                params = {
                    "rno": rno,
                    "jcd": f"{jcd:02d}",
                    "hd": d.strftime("%Y%m%d"),
                }
                dates.append(urlencode(params))
        d += timedelta(days=1)
        
    return dates

def make_url(kind, date):
    return f"https://www.boatrace.jp/owpc/pc/race/{kind}?{date}"

# main

In [9]:
import time

t0 = time.perf_counter()

dates = make_date(200)
dates = dates[:10]
paths = [make_url('racelist', D) for D in dates]
RACELIST = await main(paths)

odds_names = ["oddstf", 'odds3t', 'odds3f', 'odds2tf', 'oddsk']
paths = [[make_url(name, D) for name in odds_names] for D in dates]
ODDS = [await main(path) for path in paths]

paths = [make_url('raceresult', D) for D in dates]
RACERESULT = await main(paths)

t1 = time.perf_counter()
print(f"{t1 - t0:.3f} sec")

1.520 sec


In [10]:
X = {}
for racelist, odds, raceresult, D in zip(RACELIST, ODDS, RACERESULT, dates):
    if not racelist: continue
    data = {}
    data['racelist'] = get_racelist(racelist[3:])
    data['odds'] = odds_total(odds)
    if data['odds']==False:
        continue
    data['raceresult'] = get_result(raceresult[3:])
    X[D] = data
    # print(D)

In [11]:
import pickle

pickle.dump(X, open("boatrace_input.pkl", "wb"))